### RAG Pipeline - Data Ingestion to Vector Database Pipeline

In [ ]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
### Read all PDF files inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    #FInd all the PDF files recursively in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file.name}")
        try:
            # Use PyPDFLoader to load the PDF file
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Successfully loaded {len(documents)} pages from {pdf_file.name}.")
        except Exception as e:
            print(f"Error loading {pdf_file.name}: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

    

In [ ]:
all_pdf_documents

In [ ]:
### Text splitting get into chinks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap, #Chunk overlap means the number of characters that should overlap between consecutive chunks
        length_function = len,
        separators=["\n\n", "\n", " ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    
    #Show example of a chunk 
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks


### Embedding and Vector Database Pipeline for RAG (Retrieval-Augmented Generation) using LangChain .# 


In [ ]:
import numpy as np
from sentence_transformers import  SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        
        """
        Initialize the EmbeddingManager.
        
        Args:
            model_name : Hugging Face model name for sentence embeddings.        
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model."""
        
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        
        Args:
            texts : List of strings to embed.

        Returns:
            numpy array of embeddings with shape (n_texts, embedding_dim).
            
        """
        
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully with shape: {embeddings.shape}")

        return embeddings
    
    
    
embedding_manager = EmbeddingManager()
embedding_manager
    

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

login()  


### Vector Database Pipeline for RAG (Retrieval-Augmented Generation) using LangChain.

In [ ]:
import os
import uuid
import numpy as np
import chromadb
from chromadb.config import Settings

In [ ]:
class VectorStore:
    """manages document embeddings in a chromadb vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore.
        
        Args:
            collection_name : Name of the ChromaDB collection to store embeddings.
            persist_directory : Directory to persist the vector store.
        """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection 
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG."}
                )
            print(f"Vector store intialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")  # Show the number of existing documents in the collection
            print("ChromaDB client and collection initialized successfully.")
        except Exception as e:
            print(f"Error initializing ChromaDB client: {e}")
            raise
        
        
        
    def add_documents(self,documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.
        
        Args:
            documents : List of Langchain document.
            embeddings : Corresponding embeddings for the documents.
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        
        
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            #generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #Prepare the metadata
            metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
            metadata['doc_index'] = i 
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_texts.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for ChromaDB
            
            
            
          
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f" Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore = VectorStore()
vectorstore

In [ ]:
chunks 

In [ ]:
### Convert the text to embeddings and store in the vector database
texts = [doc.page_content for doc in chunks]

## Generate the embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)


### Retriever Pipeline from VectorStore 

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    # 1. Correct conversion from L2 Distance to Cosine Similarity
                    # similarity_score = 1 - (distance ** 2) / 2
    
                        # 2. Alternative fallback if your embeddings are not normalized:
                    similarity_score = 1 / (1 + distance) # Bounds it strictly between 0 and 1
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever

In [ ]:
print(vectorstore.collection.count())

In [ ]:
rag_retriever.retrieve("Data Ingestion & Semantic Chunking")

### Integration Vectordb Context pipline with LLM output 


In [ ]:
### Simple RAG pipeline with Groq LLM

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# 1. Import HumanMessage so Groq receives the correct structured payload
from langchain_core.messages import HumanMessage

load_dotenv()  # Load environment variables from .env file

### Initialize the Groq LLM client
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please set it in your .env file.")

llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

## Simple RAG function :  Retrieve context + Generate response using Groq LLM

def rag_simple(query, retriver, llm, top_k=3):
    ## Retriever the context from the vector store
    results = retriver.retrieve(query, top_k=top_k)
    context: str = " ".join([doc['content'] for doc in results]) if results else ""
    print(f"Retrieved context: \n{context}\n")
    if not context:
        return "No relevant context found for the query."
    
    ## Generate the response using Groq LLM
    # 2. Build the final prompt directly using the variables
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    # 3. Pass the prompt inside a HumanMessage object
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
answer = rag_simple("Data Ingestion & Semantic Chunking", rag_retriever, llm)
print(f"Answer: {answer}")

### Enhanced the RAG pipeline features 

In [ ]:
# --- Enhanced RAG pipeline feature ---

def rag_advanced(query, retriever, llm, top_k =5, min_score = 0.2, return_context = False):
    """
    RAG pipeline with extra features:
    
    - Returns answer, sources, confidence score, and optionally full context.
    
    """
    
    results = retriever.retrieve(query, top_k=top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    #Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page','unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120]+'...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    
    #Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
    }
    

    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Idiomatic LCEL Implementation", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context preview:", result['context'][:300])



### Advanced RAG pipeline : Streaming, Citations, History, Summarization ----

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(prompt)
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Concurrency Models: Threading, Multiprocessing, and Asyncio", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])
        
